In [1]:
import pandas as pd
import numpy as np

In [5]:
#시간을 변환하고 요일 편향을 
orders = pd.read_csv('../data/orders.csv')

In [ ]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        200000 non-null  int64         
 1   customer_id     200000 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         200000 non-null  str           
 4   status          200000 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 9.7 MB


In [6]:
print(f'order_datetime 변환전 타입: {orders["order_datetime"].dtype}')

order_datetime 변환전 타입: str


In [7]:
orders['order_datetime'] = pd.to_datetime(
    orders['order_datetime'], errors='coerce'
)

In [8]:
# '2026-07-24 14:24:50' => 정상 변환 가능
# '206-07-24 14:24:50' => 정상 변환 불가능 -> 에러 -> NaT errors = 'coerce'
print(f'orders_datetime 변환 후 데이터 타입: {orders["order_datetime"].dtype}')

orders_datetime 변환 후 데이터 타입: datetime64[us]


In [ ]:
#NaT(== NaN, Null) 확인
orders['order_datetime'].isna().sum()  #1933개의 null

np.int64(1933)

In [ ]:
#2024년 6월 이후의 주문 건수
june_on = orders[orders['order_datetime'] >= '2024-06-01']
june_on.head()

#shape[]  건수
june_on.shape[0]

42959

In [14]:
orders['order_datetime']

0        2024-06-25 00:17:41
1        2024-05-28 19:35:20
2        2024-02-14 17:49:14
3        2024-05-08 16:37:36
4        2024-04-17 20:08:22
                 ...        
199995   2024-03-26 08:48:07
199996   2024-01-25 18:46:24
199997   2024-01-20 06:44:00
199998   2024-03-21 00:22:40
199999   2024-01-01 00:58:02
Name: order_datetime, Length: 200000, dtype: datetime64[us]

In [ ]:
#min이랑 max
print(f'최초 주문: {orders["order_datetime"].min()}')
# print(f'최초 주문: {orders["order_datetime"].min()}')

print(f'최후 주문: {orders["order_datetime"].max()}')



최후 주문: 2025-06-30 03:01:14


In [18]:
#dt 접근자: 시간 축 추출, 요일별 주문 개수

#1930개 정도의 null을 dropna로
#중요한 column의 NaN을 지워야됨
orders_df=orders.dropna(subset="order_datetime") #삭제

In [22]:
#추출
orders_df['month'] = orders_df['order_datetime'].dt.month
orders_df['dow'] = orders_df['order_datetime'].dt.dayofweek
orders_df['dow_name']=orders_df['order_datetime'].dt.day_name()

In [23]:
orders_df.info()

<class 'pandas.DataFrame'>
Index: 198067 entries, 0 to 199999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        198067 non-null  int64         
 1   customer_id     198067 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         198067 non-null  str           
 4   status          198067 non-null  str           
 5   month           198067 non-null  int32         
 6   dow             198067 non-null  int32         
 7   dow_name        198067 non-null  str           
dtypes: datetime64[us](1), int32(2), int64(2), str(3)
memory usage: 15.6 MB


In [24]:
orders_df.head()

,order_id,customer_id,order_datetime,channel,status,month,dow,dow_name
0,25463,1077,2024-06-25 00:17:41,store,delivered,6,1,Tuesday
1,171088,2998,2024-05-28 19:35:20,web,delivered,5,1,Tuesday
2,27437,4066,2024-02-14 17:49:14,app,delivered,2,2,Wednesday
3,98425,3243,2024-05-08 16:37:36,store,delivered,5,2,Wednesday
4,22325,5331,2024-04-17 20:08:22,app,delivered,4,2,Wednesday


In [30]:
#요일별 주문 건수
# orders_df.groupby('dow_name') #DataFrameGroupBy
# orders_df.groupby('dow_name')['order_id'] #SeriesGroupBy 
# orders_df.groupby('dow_name')['order_id'].count()

#요일명의 정렬 사전 작업
order_names =  ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', "Sunday"]
orders_df.groupby('dow_name')['order_id'].count().reindex(order_names)

dow_name
Monday       25286
Tuesday      25134
Wednesday    24977
Thursday     25568
Friday       25478
Saturday     36037
Sunday       35587
Name: order_id, dtype: int64

In [27]:
type(orders_df)

pandas.DataFrame

In [41]:
orders_df['order_datetime']

0        2024-06-25 00:17:41
1        2024-05-28 19:35:20
2        2024-02-14 17:49:14
3        2024-05-08 16:37:36
4        2024-04-17 20:08:22
                 ...        
199995   2024-03-26 08:48:07
199996   2024-01-25 18:46:24
199997   2024-01-20 06:44:00
199998   2024-03-21 00:22:40
199999   2024-01-01 00:58:02
Name: order_datetime, Length: 198067, dtype: datetime64[us]

In [40]:
#일평균 환산
#일별 평균 환산

#normalize() 하면 시분초가 없어짐 (00:00:00)
#일별 하려면 (일별 group by) 시분초를 없애야함 왜냐하면 시분초로 group by 의미가 없기 때문
days =orders_df['order_datetime'].dt.normalize()  #series
# type(days)  #series

In [ ]:
# .size()가 날짜별 행 개수를 세었다
day_counts = orders_df.groupby(days).size()
day_counts #series



order_datetime
2024-01-01    639
2024-01-02    687
2024-01-03    696
2024-01-04    677
2024-01-05    702
             ... 
2025-06-20      1
2025-06-22      1
2025-06-23      1
2025-06-24      1
2025-06-30      1
Length: 222, dtype: int64

In [46]:
#주말은 day of week (dow) >= 5

#주말인지 주중인지 판단 -> True, False로 나옴
is_weekend = day_counts.index.dayofweek >= 5


In [ ]:
day_counts[is_weekend].mean()  #1119.125

np.float64(1119.125)

In [ ]:
#평일 평균 주문건수
day_counts[~is_weekend].mean() # 800.272151898734  800건은 평일

np.float64(800.2721518987341)

In [51]:
#시간대별 주문 건수 : 언제(시간단위) 가장 많이 주문하나
orders_df['hour'] = orders_df['order_datetime'].dt.hour
# orders_df['hour'][:10]
by_hour = orders_df.groupby('hour')['order_id'].count()
by_hour

hour
0     8239
1     8239
2     8210
3     8216
4     8318
5     8157
6     8109
7     8355
8     8264
9     8182
10    8319
11    8264
12    8403
13    8326
14    8285
15    8278
16    8102
17    8309
18    8243
19    8212
20    8358
21    8235
22    8156
23    8288
Name: order_id, dtype: int64

In [56]:
# 가장 많은 건수: max
print(f'가장 많은 건수 : {by_hour.max()}') # 8403  <- 가장 많은 건수 

#가장 많은 건수가 있는 시간대를 알아보려면?
#최댓값의 index를 가져오는 함수
print(f'가장 많은 건수 index : {by_hour.idxmax()}')  #12


가장 많은 건수 : 8403
가장 많은 건수 index : 12


In [ ]:
#가입일 (customers)부터 첫 주문(orders - orders_df)까지 걸린 일수
# merge안하고 하려면?
# orders는 order_df
 
customers_df = pd.read_csv('../data/customers.csv')
customers_df.info()   #  signup_date  5000 non-null   <- 문자열이므로 datetime으로 바꾸자

#5000명이 주문    1번이 주문을 3번하면? <- 최초 주문은 그 3번중의 최솟값


<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  5000 non-null   int64
 1   name         5000 non-null   str  
 2   gender       4696 non-null   str  
 3   birth_date   4616 non-null   str  
 4   signup_date  5000 non-null   str  
 5   city         5000 non-null   str  
 6   email        4752 non-null   str  
dtypes: int64(1), str(6)
memory usage: 544.3 KB


In [ ]:
#signup_date 컬럼의 데이터 타입을 datetime로 변경
customers_df['signup_date'] = pd.to_datetime(
    customers_df['signup_date'], errors = 'coerce'
)

#고객별 최초 주문 날짜
first_order = orders_df.groupby('customer_id')['order_datetime'].min()
first_order   #customer_id 가 index로 들어감
# Length: 5941건


customer_id
1000     2024-01-21 23:31:35
1001     2024-01-03 12:01:52
1002     2024-01-06 10:37:15
1003     2024-01-03 09:42:56
1004     2024-01-01 04:23:36
                 ...        
989772   2024-01-10 04:09:52
989922   2024-06-29 03:27:02
989960   2024-06-09 16:13:18
989977   2024-01-14 00:14:02
989995   2024-05-27 01:54:32
Name: order_datetime, Length: 5941, dtype: datetime64[us]

In [61]:
customers_df.drop_duplicates('customer_id')

,customer_id,name,gender,birth_date,signup_date,city,email
0,3293,권준서,여,1954-04-29,2020-04-26,부산,user3293@example.com
1,1862,박준선,F,1980-05-19,2023-09-01,서울,user1862@example.com
2,4955,전영경,여,1980-06-08,2024-04-30,인천,user4955@example.com
3,4653,한재하,M,1971-06-09,2023-08-06,서울,user4653@example.com
4,3331,송준연,여,1971-02-01,2023-12-20,대전,user3331@example.com
...,...,...,...,...,...,...,...
4995,5672,정우은,M,1956-09-22,2023-10-28,서울,user5672@example.com
4996,3365,이예재,M,1991-06-10,2024-03-31,고양,user3365@example.com
4997,5176,조아준,M,1967-04-29,2021-03-05,인천,user5176@example.com
4998,3915,장성우,F,1996-08-27,2022-11-12,대전,NaN


<bound method Series.min of 0         1077
1         2998
2         4066
3         3243
4         5331
          ... 
199995    3442
199996    5765
199997    3265
199998    2403
199999    3439
Name: customer_id, Length: 198067, dtype: int64>

In [66]:
customers_df['customer_id'].max()

np.int64(5949)

In [67]:
orders_df['customer_id'].max()

np.int64(989995)

In [ ]:
cust = customers_df.drop_duplicates('customer_id').set_index('customer_id')
cust.head()

# 	name	gender	birth_date	signup_date	city	email
# customer_id						
# 3293	권준서	여	1954-04-29	2020-04-26	부산	user3293@example.com
# 1862	박준선	F	1980-05-19	2023-09-01	서울	user1862@example.com
# 4955	전영경	여	1980-06-08	2024-04-30	인천	user4955@example.com
# 4653	한재하	M	1971-06-09	2023-08-06	서울	user4653@example.com
# 3331	송준연	여	1971-02-01	2023-12-20	대전	user3331@example.com


,name,gender,birth_date,signup_date,city,email
customer_id,,,,,,
3293,권준서,여,1954-04-29,2020-04-26,부산,user3293@example.com
1862,박준선,F,1980-05-19,2023-09-01,서울,user1862@example.com
4955,전영경,여,1980-06-08,2024-04-30,인천,user4955@example.com
4653,한재하,M,1971-06-09,2023-08-06,서울,user4653@example.com
3331,송준연,여,1971-02-01,2023-12-20,대전,user3331@example.com


In [71]:
type(first_order)

pandas.Series

In [ ]:
#.set_index()를 사용해서 인덱스를 뺴고 -> 인덱스 라벨 매칭 해서 같은 인덱스끼리 빼려고 
# 최초 주문일 - 회원가입일
# = 가입 후 첫 주문까지 걸린 기간

In [ ]:
#series - series =>  Series - Series 연산은 단순히 같은 위치끼리 계산하는 게 아니라, 먼저 인덱스 라벨을 맞춘 다음 계산해.
gap =  (first_order - cust['signup_date']).dt.days.dropna()
gap.mean()   #가입 후 최초 주문 평균 일수 

#이렇게 빼면 timeDelta 가 나오는데 dt를 쓸 수 있다
#dt를 쓰면 년,월, 일, 시분초, 일자를 뽑을 수 있고 그래서 dt.days 

customer_id
1000     205.0
1001     214.0
1002     800.0
1003    -152.0
1004    -158.0
1005    1507.0
1006     530.0
1007    1152.0
1008    -175.0
1009    -117.0
dtype: float64